# Model development: classifier families, overfitting check and timings

Reads the numeric feature matrix saved by `data_preprocessing.ipynb` from `data/process/`. Every
feature column is already numeric, one-hot encoded and scaled, so the models read it as is. Six
families:

- linear: LinearSVC, logistic regression, SGD (modified Huber), ridge classifier (sklearn)
- boosting: XGBoost, CatBoost (libraries outside sklearn)

Optuna maximises 5-fold stratified CV macro-F1 on the training rows. The test set has no labels, so
the best settings are refit on all of train and used to predict test.

Scoring is macro-F1: every category counts equally, so small categories matter as much as large ones.

The TF-IDF settings (character n-grams 2 to 5, min_df 2) are fixed in preprocessing, so the search
tunes only the classifier.

In [ ]:
import json
import time
from collections import Counter
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import LinearSVC
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

optuna.logging.set_verbosity(optuna.logging.WARNING)

DATA = Path("..") / "data"  # notebook lives in notebooks/
PROCESS = DATA / "process"
N_THREADS = 8  # threads inside each boosting model (folds run one after another, so timings are not contended)
TARGET = "category"

train = pd.read_csv(PROCESS / "train_features.csv")
test = pd.read_csv(PROCESS / "test_features.csv")
print("train", train.shape, "| test", test.shape)

## 1. Model inputs

All columns except `id` and the `category` label are features. They are already numeric, so no
transformation happens here.

In [ ]:
FEATURE_COLS = [c for c in train.columns if c not in ("id", TARGET)]
# Some boosting libraries reject feature names with special characters (TF-IDF n-grams contain them),
# so the columns get plain names. FEATURE_COLS keeps the original names for reference.
SHORT_NAMES = [f"f{i}" for i in range(len(FEATURE_COLS))]
X = train[FEATURE_COLS].astype(float).set_axis(SHORT_NAMES, axis=1)
y = train[TARGET]
X_test = test[FEATURE_COLS].astype(float).set_axis(SHORT_NAMES, axis=1)
CLASSES = sorted(y.unique())
print("features:", len(FEATURE_COLS), "| classes:", len(CLASSES), "| smallest class:", y.value_counts().min(), "rows")

## 2. Scoring and CV

The competition metric: unweighted mean of per-category F1. `labels` fixes the set to all
categories, so a category the model never predicts scores F1 = 0 rather than being left out.

In [ ]:
def macro_f1(y_true, y_pred) -> float:
    return f1_score(y_true, y_pred, average="macro", labels=CLASSES, zero_division=0)


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

## 3. Classifier families

Boosting libraries do not all accept string labels or sklearn's `class_weight`, so they are wrapped:
labels are encoded to integers, and class balance is applied as balanced sample weights.

In [ ]:
class EncodedBooster(BaseEstimator, ClassifierMixin):
    """Wraps a boosting estimator: integer labels and balanced sample weights."""

    def __init__(self, estimator=None, balanced=True):
        self.estimator = estimator
        self.balanced = balanced

    def fit(self, X, y):
        self.label_encoder_ = LabelEncoder().fit(y)
        y_enc = self.label_encoder_.transform(y)
        weight = compute_sample_weight("balanced", y_enc) if self.balanced else None
        self.model_ = clone(self.estimator).fit(X, y_enc, sample_weight=weight)
        self.classes_ = self.label_encoder_.classes_
        return self

    def predict(self, X):
        pred = np.asarray(self.model_.predict(X)).ravel().astype(int)
        return self.label_encoder_.inverse_transform(pred)


FAMILIES = ["linear_svc", "logistic", "sgd", "ridge", "xgboost", "catboost"]
DEFAULT_PARAMS = {"family": "linear_svc", "class_weight": "balanced", "C": 0.5}


def build_model(params: dict):
    p = dict(params)
    family = p["family"]
    balanced = p["class_weight"] == "balanced"
    if family == "linear_svc":
        return LinearSVC(C=p["C"], class_weight=p["class_weight"], max_iter=5000)
    if family == "logistic":
        return LogisticRegression(C=p["C"], class_weight=p["class_weight"], max_iter=2000)
    if family == "sgd":
        return SGDClassifier(loss="modified_huber", alpha=p["alpha"], class_weight=p["class_weight"],
                             max_iter=2000, tol=1e-4, random_state=0)
    if family == "ridge":
        return RidgeClassifier(alpha=p["alpha"], class_weight=p["class_weight"])
    if family == "xgboost":
        return EncodedBooster(
            XGBClassifier(n_estimators=p["n_estimators"], learning_rate=p["learning_rate"],
                          max_depth=p["max_depth"], colsample_bytree=p["colsample_bytree"],
                          subsample=p["subsample"], min_child_weight=p["min_child_weight"],
                          tree_method="hist", max_bin=64, n_jobs=N_THREADS, random_state=0),
            balanced=balanced)
    if family == "catboost":
        return EncodedBooster(
            CatBoostClassifier(iterations=p["iterations"], depth=p["depth"], learning_rate=p["learning_rate"],
                               l2_leaf_reg=p["l2_leaf_reg"], rsm=0.05, border_count=16,
                               thread_count=N_THREADS, random_seed=0, verbose=0, allow_writing_files=False),
            balanced=balanced)
    raise ValueError(f"unknown family: {family}")

## 4. Cross-validation with train scores and timings

Each fold is fitted, then scored on its training rows (to measure fit) and its held-out rows (to
measure generalisation). Fit and predict times are recorded per fold. Folds run one after another,
so the times are not affected by other folds running at the same time.

In [ ]:
def run_cv(params: dict) -> dict:
    oof = np.empty(len(y), dtype=object)
    train_f1, val_f1, fit_s, pred_s = [], [], [], []
    for tr_idx, va_idx in cv.split(X, y):
        model = build_model(params)
        t0 = time.perf_counter()
        model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        fit_s.append(time.perf_counter() - t0)
        t0 = time.perf_counter()
        va_pred = model.predict(X.iloc[va_idx])
        pred_s.append(time.perf_counter() - t0)
        oof[va_idx] = va_pred
        train_f1.append(macro_f1(y.iloc[tr_idx], model.predict(X.iloc[tr_idx])))
        val_f1.append(macro_f1(y.iloc[va_idx], va_pred))
    return {
        "oof_f1": macro_f1(y, oof),  # pooled out-of-fold score, the one used for tuning
        "val_f1_mean": float(np.mean(val_f1)),
        "val_f1_sd": float(np.std(val_f1)),
        "train_f1_mean": float(np.mean(train_f1)),
        "fit_s": float(np.mean(fit_s)),
        "predict_s": float(np.mean(pred_s)),
        "oof_pred": oof,
    }


default_result = run_cv(DEFAULT_PARAMS)
print(f"default (LinearSVC): pooled CV macro-F1 = {default_result['oof_f1']:.4f}")

## 5. Optuna search over family and settings

Each trial picks a family, then that family's settings. The default LinearSVC settings are queued first.
The sampler is seeded, so the run repeats. Each trial stores its train and validation scores and timings.

In [ ]:
N_TRIALS = 30  # boosting trials are slow (catboost about 2 minutes each), so the search is kept short


def objective(trial: optuna.Trial) -> float:
    family = trial.suggest_categorical("family", FAMILIES)
    params = {
        "family": family,
        "class_weight": trial.suggest_categorical("class_weight", ["balanced", None]),
    }
    if family == "linear_svc":
        params["C"] = trial.suggest_float("linear_svc_C", 0.01, 10.0, log=True)
    elif family == "logistic":
        params["C"] = trial.suggest_float("logistic_C", 0.1, 100.0, log=True)
    elif family == "sgd":
        params["alpha"] = trial.suggest_float("sgd_alpha", 1e-6, 1e-2, log=True)
    elif family == "ridge":
        params["alpha"] = trial.suggest_float("ridge_alpha", 1e-2, 10.0, log=True)
    elif family == "xgboost":
        params.update(
            n_estimators=trial.suggest_int("xgboost_n_estimators", 100, 300),
            learning_rate=trial.suggest_float("xgboost_learning_rate", 0.03, 0.3, log=True),
            max_depth=trial.suggest_int("xgboost_max_depth", 3, 6),
            colsample_bytree=trial.suggest_float("xgboost_colsample", 0.3, 1.0),
            subsample=trial.suggest_float("xgboost_subsample", 0.6, 1.0),
            min_child_weight=trial.suggest_int("xgboost_min_child_weight", 1, 10),
        )
    else:  # catboost
        params.update(
            iterations=trial.suggest_int("catboost_iterations", 150, 300),
            depth=trial.suggest_int("catboost_depth", 4, 6),
            learning_rate=trial.suggest_float("catboost_learning_rate", 0.03, 0.3, log=True),
            l2_leaf_reg=trial.suggest_float("catboost_l2_leaf_reg", 1.0, 10.0, log=True),
        )
    result = run_cv(params)
    trial.set_user_attr("model_params", params)
    for key in ("val_f1_mean", "val_f1_sd", "train_f1_mean", "fit_s", "predict_s"):
        trial.set_user_attr(key, result[key])
    return result["oof_f1"]


study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=0))
study.enqueue_trial({"family": "linear_svc", "class_weight": "balanced", "linear_svc_C": 0.5})
study.optimize(objective, n_trials=N_TRIALS)

best_model_params = study.best_trial.user_attrs["model_params"]
print("trials:", len(study.trials), "| best CV macro-F1:", round(study.best_value, 4))
print("best settings:", best_model_params)

## 6. Overfitting check and timings

The best trial of each family, plus the default LinearSVC.

- **gap** = train macro-F1 minus validation macro-F1, averaged over folds. A large gap means the model
  fits the training rows much better than rows it has not seen.
- Linear models on TF-IDF often reach near-perfect train scores, so a large gap is not proof of
  overfitting on its own. Compare the families against each other, and check whether a higher gap
  buys a higher validation score.
- **verdict** uses fixed thresholds on the gap: 0.10 or more "overfits", 0.05 to 0.10 "some overfit".
- **fit s / predict s** are seconds per fold. **trial s** is the mean Optuna trial duration for that family
  (five folds, including scoring).

In [ ]:
GAP_OVERFIT, GAP_SOME = 0.10, 0.05


def verdict(gap: float) -> str:
    if gap >= GAP_OVERFIT:
        return "overfits"
    if gap >= GAP_SOME:
        return "some overfit"
    return "no clear overfit"


trials_by_family = Counter()
trial_seconds = {}
best_by_family = {}
for t in study.trials:
    if t.state != optuna.trial.TrialState.COMPLETE:
        continue
    fam = t.user_attrs["model_params"]["family"]
    trials_by_family[fam] += 1
    trial_seconds.setdefault(fam, []).append(t.duration.total_seconds())
    if fam not in best_by_family or t.value > best_by_family[fam].value:
        best_by_family[fam] = t


def summary_row(label: str, result: dict, trials: int, seconds: float) -> dict:
    gap = result["train_f1_mean"] - result["val_f1_mean"]
    return {
        "model": label,
        "cv_macro_f1": round(result["oof_f1"], 4),
        "val_fold_mean": round(result["val_f1_mean"], 4),
        "val_fold_sd": round(result["val_f1_sd"], 4),
        "train_fold_mean": round(result["train_f1_mean"], 4),
        "gap": round(gap, 4),
        "verdict": verdict(gap),
        "fit_s": round(result["fit_s"], 2),
        "predict_s": round(result["predict_s"], 3),
        "trials": trials,
        "trial_s": round(seconds, 1),
    }


rows = []
for fam, t in best_by_family.items():
    a = t.user_attrs
    rows.append(summary_row(
        fam, {"oof_f1": t.value, "val_f1_mean": a["val_f1_mean"], "val_f1_sd": a["val_f1_sd"],
              "train_f1_mean": a["train_f1_mean"], "fit_s": a["fit_s"], "predict_s": a["predict_s"]},
        trials_by_family[fam], float(np.mean(trial_seconds[fam]))))
rows.append(summary_row("linear_svc (default)", default_result, 0, float("nan")))

summary = pd.DataFrame(rows).sort_values("cv_macro_f1", ascending=False).reset_index(drop=True)
PROCESS.mkdir(parents=True, exist_ok=True)
summary.to_csv(PROCESS / "model_summary.csv", index=False)
summary

## 7. Tuned model: per-class results

Out-of-fold predictions from the best overall settings. Per-class F1 shows which categories are weakest.

In [ ]:
(PROCESS / "best_params.json").write_text(json.dumps(best_model_params, indent=2))
print("saved", PROCESS / "best_params.json")

best_result = run_cv(best_model_params)
print(f"best ({best_model_params['family']}): pooled CV macro-F1 = {best_result['oof_f1']:.4f}")

report = classification_report(y, best_result["oof_pred"], output_dict=True, zero_division=0)
per_class = pd.DataFrame({k: v for k, v in report.items() if isinstance(v, dict)}).T
per_class.sort_values("f1-score").head(10).round(3)

## 8. Refit on all training rows and predict test

The test file has no labels. The predictions are checked against the submission format (`id`,
`category`) and the ids are checked against `sample_submission.csv`.

In [ ]:
tuned = build_model(best_model_params)
tuned.fit(X, y)
test_pred = tuned.predict(X_test)
submission = pd.DataFrame({"id": test["id"], "category": test_pred})

expected_ids = set(pd.read_csv(DATA / "sample_submission.csv")["id"])
assert set(submission["id"]) == expected_ids, "test ids do not match sample_submission.csv"
assert submission["category"].isin(set(y)).all(), "predicted a category not in train"

submission.to_csv(PROCESS / "test_predictions.csv", index=False)
print("saved", PROCESS / "test_predictions.csv", submission.shape)
submission["category"].value_counts().head(10)